# Week 8, Day 2: Build a Full RAG System

In Day 1 you learned what RAG is. Today you build one, the whole system,
with your own hands. By the end of this notebook you will have a working
question answering system that reads your own course notes and answers
questions about them.

## What you are building

A full RAG system has five parts, and you will build all five:

1. **Corpus.** The documents the system can read. Today that is your own
course material: every concept guide and notebook from Week 1 to Week 8,
as plain text files.
2. **Chunking.** Cutting the documents into small pieces.
3. **Embeddings.** Turning each piece into a vector of numbers.
4. **Vector database.** Storing those vectors so you can search them fast.
Today the vector database is FAISS.
5. **Generation.** Sending the best pieces plus your question to DeepSeek,
which writes the answer.

## What you need

- This notebook running on Kaggle, with internet turned on.
- The course documents folder, called `rag_course_docs`, uploaded next to
this notebook. It holds 13 text files, one per topic.
- Your Hugging Face token, which you will type in when asked. It is never
written into the notebook.

## How to work through this notebook

Read the explanation before each code cell, then run the cell and read its
output before moving on. Every step prints what it did, so you can see the
system working. If an output looks wrong, stop there and fix it before
moving on. That is the whole skill.

## What is a vector database?

A vector database is a database built for one job: store vectors, and find
the ones closest to a query vector, fast.

You already know the idea from Day 1. An embedding turns a piece of text
into a list of 384 numbers, a point in a 384 dimensional space. Similar
meanings end up near each other. So "find the most relevant chunk" becomes
"find the nearest points to the question vector". A vector database does
exactly that search, and it does it fast even when there are millions of
points.

Why not just keep the vectors in a Python list and compare them one by one?
For 1,342 chunks, a plain list works fine, and you did that kind of math in
Day 1. But at a million chunks, comparing the query against every single
vector takes too long. A vector database organizes the vectors so most of
them never need to be compared at all. It also saves the vectors to disk,
so the database survives after the notebook closes. That is what makes it a
database instead of a variable.

Today you will use **FAISS**. There are others, and they are listed later
in this notebook, but FAISS is the one most RAG systems start with.

In [32]:
# Step 0a: make sure FAISS is available (run on Kaggle)
try:
    import faiss
    print('FAISS is already installed, version', faiss.__version__)
except ImportError:
    print('FAISS is not installed, installing it now...')
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
    import faiss
    print('FAISS installed, version', faiss.__version__)

FAISS is already installed, version 1.15.1


In [33]:
# Step 0b: load the corpus (your own course documents)
import pathlib

candidates = [
    pathlib.Path('/kaggle/input/datasets/yousafkhatana/rag-files'),
    
]
corpus_dir = None
for path in candidates:
    if path.exists() and list(path.glob('*.txt')):
        corpus_dir = path
        break
print('corpus folder:', corpus_dir)

docs = {}
for p in sorted(corpus_dir.glob('*.txt')):
    docs[p.name] = p.read_text()
for name, text in docs.items():
    print(f'{name}: {len(text):,} characters')
print(f'\nTotal: {len(docs)} files, {sum(len(t) for t in docs.values()):,} characters')

corpus folder: /kaggle/input/datasets/yousafkhatana/rag-files
01_ai_fundamentals.txt: 11,703 characters
02_huawei_ai_ecosystem.txt: 13,714 characters
03_requirement_analysis.txt: 22,241 characters
04_specification_solution_design.txt: 31,383 characters
05_fine_tuning_generative_ai.txt: 9,561 characters
06_rag_concepts.txt: 27,672 characters
07_python_foundations.txt: 21,757 characters
08_numpy_pandas.txt: 21,283 characters
09_ml_scikit_learn.txt: 30,389 characters
10_huggingface_deepseek.txt: 18,574 characters
11_computer_vision_ocr.txt: 18,922 characters
12_evaluation_metrics.txt: 37,084 characters
13_sms_spam_finetuning.txt: 12,605 characters

Total: 13 files, 276,888 characters


## Chunking: why you cannot embed whole documents

An embedding model reads a limited amount of text at a time. Our model,
all-MiniLM-L6-v2, reads at most 256 tokens, roughly 1,000 characters. A
whole guide is 10,000 to 30,000 characters, so it does not fit. And even if
it did fit, one vector for a whole guide would be useless: the guide talks
about ten different things, and their meanings would all blur into one
point.

So you cut each document into chunks, small pieces of about 300 characters
each. Each chunk gets its own vector. Retrieval then finds the specific
piece that answers the question, not the whole document.

Two settings control chunking:

- **Chunk size.** How long each piece may be. 300 characters is about two
or three sentences. Small enough to be about one idea, big enough to carry
context.
- **Chunk overlap.** How much of the next chunk is repeated at the end of
the previous one. 50 characters here. Overlap is insurance: if a sentence
is cut at the boundary between two chunks, it still appears whole in one
of them.

## Recursive chunking

Fixed size chunking cuts blindly every 300 characters, even in the middle
of a word. Recursive chunking is smarter. It tries big separators first,
and only cuts at smaller ones when it has to.

The separators, in order, are:

1. `"\n\n"`, a blank line, which is a paragraph break.
2. `"\n"`, a single line break.
3. `". "`, the end of a sentence.
4. `" "`, a single space, the last resort.

How it works: split the text at paragraph breaks. If a piece is still
longer than the chunk size, split that piece at line breaks. Still too
long, split at sentence ends. Still too long, split at spaces. A piece
that already fits is left alone, never cut further.

So a short paragraph stays whole. A long paragraph is cut at sentence
ends, not in the middle of a sentence. Only a very long sentence is cut at
spaces. The chunks follow the natural structure of the text instead of
fighting it.

That is why it is called recursive: the same splitting function calls
itself on any piece that is still too big, each time with the next smaller
separator.

In [34]:
# Step 1a: the recursive chunker
def recursive_chunk(text, chunk_size=300, chunk_overlap=50):
    separators = ["\n\n", "\n", ". ", " "]

    def _split(t, seps):
        if len(t) <= chunk_size:
            return [t]
        if not seps:
            step = chunk_size - chunk_overlap
            return [t[i:i + chunk_size] for i in range(0, len(t), step)]
        sep = seps[0]
        parts = t.split(sep)
        # keep the separator attached to each piece, except the last one
        pieces = [p + sep if i < len(parts) - 1 else p
                  for i, p in enumerate(parts)]
        chunks, current = [], ""
        for piece in pieces:
            if len(current) + len(piece) <= chunk_size:
                current += piece
            else:
                if current:
                    chunks.append(current)
                if len(piece) > chunk_size:
                    chunks.extend(_split(piece, seps[1:]))
                    current = ""
                else:
                    current = piece
        if current:
            chunks.append(current)
        return chunks

    chunks = [c for c in _split(text, separators) if c.strip()]
    # overlap: each chunk carries the first chunk_overlap characters
    # of the next chunk, so a sentence cut at a boundary still
    # appears whole in one of the two chunks
    out = []
    for i, c in enumerate(chunks):
        if i < len(chunks) - 1:
            out.append(c + chunks[i + 1][:chunk_overlap])
        else:
            out.append(c)
    return out

# quick check on one short text before running it on the whole corpus
test = ("First sentence here. Second sentence here.\n\nNew paragraph "
        "starts here. It keeps going for a while.")
print(recursive_chunk(test, chunk_size=60, chunk_overlap=10))








# Step 1b: chunk the whole corpus
chunks, sources = [], []
for name, text in docs.items():
    for c in recursive_chunk(text):
        chunks.append(c)
        sources.append(name)

print(f'Documents: {len(docs)}')
print(f'Chunks: {len(chunks)}')
print(f'Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.1f} characters')
print(f'Longest chunk: {max(len(c) for c in chunks)} characters')
print(f'Shortest chunk: {min(len(c) for c in chunks)} characters')
print()
print('--- Example chunk 1 ---')
print(chunks[0])
print()
print('--- Example chunk 2 ---')
print(chunks[1])

['First sentence here. Second sentence here.\n\nNew paragr', 'New paragraph starts here. It keeps going for a while.']
Documents: 13
Chunks: 1342
Average chunk length: 254.7 characters
Longest chunk: 350 characters
Shortest chunk: 8 characters

--- Example chunk 1 ---
AI Fundamentals
History, Types & the Global
Landscape
Day 1, Concept Guide

Before you write a line of code or open a single H

--- Example chunk 2 ---
Before you write a line of code or open a single Huawei Cloud service, you need the vocabulary.
What actually counts as AI? Where did it come from? And why is nearly every major company on
earth racing to adopt it right now? Today answers all three, and gives you the shared language the
rest of this course builds on.




#### Step 1b: chunk the whole corpus
chunks, sources = [], []
for name, text in docs.items():
    for c in recursive_chunk(text):
        chunks.append(c)
        sources.append(name)

print(f'Documents: {len(docs)}')
print(f'Chunks: {len(chunks)}')
print(f'Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.1f} characters')
print(f'Longest chunk: {max(len(c) for c in chunks)} characters')
print(f'Shortest chunk: {min(len(c) for c in chunks)} characters')
print()
print('--- Example chunk 1 ---')
print(chunks[0])
print()
print('--- Example chunk 2 ---')
print(chunks[1])

## The split chunk problem

Chunking has one weakness you must know about. When a single fact is split
across two chunks, retrieval may return only one half, and the model
answers from half the facts.

You met this in Day 1. Two facts far apart in a document: the shop's Sunday
hours on page 1 and the cake return rule on page 5. If the question needs
both facts, both chunks must reach the top-k. If only one does, the answer
is incomplete.

Day 1 gave five fixes for split information. They are worth repeating here
because you will use them in real systems:

1. **Raise k.** Retrieve more chunks, so the second half has room to enter.
2. **Overlap, or larger chunks.** A bigger window is less likely to cut a
fact in half.
3. **Parent document retrieval.** Retrieve the small chunk, but send its
whole parent document to the model.
4. **Contextual chunks.** Prepend the section heading to each chunk, so a
chunk carries its context with it.
5. **Multi-step retrieval.** Ask a focused follow-up question to fetch the
missing half.

The next cell shows the problem live, on your own course material. One
question asks for two facts that live in two different files: the SMS spam
accuracy, which is in the fine-tuning notes, and the definition of RAG,
which is in the RAG guide. You will run that test in Step 4c, after the
index exists. Watch what retrieval returns.

## Embeddings: a quick recap

You know this from Day 1. An embedding model reads a piece of text and
returns a list of numbers, a vector, that captures its meaning. Similar
meanings get similar vectors. Our model is all-MiniLM-L6-v2, and its
vectors have 384 numbers each. You verified the number 384 yourself in
Day 1 with a real run.

Two rules from Day 1 still apply:

- The question and the chunks must go through the **same** embedding
model, or their vectors are not comparable.
- The embedding model is an encoder, the same family as DistilBERT from
Week 6. It reads text, it does not write text. Generation is a separate
job, done by DeepSeek later.

In [35]:
# Step 2: embed every chunk
# all-MiniLM-L6-v2 is a public model, so no token is needed to load it.
# It downloads from Hugging Face the first time, then it is cached.
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model.eval()

def embed(texts, batch_size=32):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=256, return_tensors='pt')
        with torch.no_grad():
            output = model(**encoded).last_hidden_state
        mask = encoded['attention_mask'].unsqueeze(-1)
        mean = (output * mask).sum(1) / mask.sum(1)
        vectors.append(mean.numpy())
    return np.vstack(vectors).astype('float32')

chunk_embeddings = embed(chunks)
print('Embeddings shape:', chunk_embeddings.shape)
print('Each chunk is now a vector of', chunk_embeddings.shape[1], 'numbers.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (1342, 384)
Each chunk is now a vector of 384 numbers.


## FAISS: the vector database

FAISS stands for Facebook AI Similarity Search. It is a library built by
Meta for searching through vectors fast. It is not a server and it does not
need an API key. You call it from Python, the same way you call NumPy.

What FAISS gives you that a plain list does not:

- **Indexes.** A data structure built for searching, not just storing.
The index you build today holds all 1,342 chunk vectors.
- **Fast search.** A search returns the top-k nearest vectors in
milliseconds, even at millions of vectors.
- **Saving and loading.** You can write the index to a file and load it
back later. The database survives after the notebook closes.

### Flat and IVF: the two indexes you must know

**Flat (IndexFlatIP, IndexFlatL2).** Flat means no structure and no
shortcuts. The index stores the vectors as they are. At search time it
compares the query against EVERY vector and returns the best ones. This is
called exact search, and its recall is perfect: it cannot miss the true
nearest neighbour, because it checked everything. The price is speed. At a
million vectors, every search scans a million vectors. Flat is the right
choice below roughly 100,000 vectors, which includes every classroom
project. Today you use Flat.

IP means inner product. When both vectors are normalized to length 1, the
inner product equals cosine similarity, which you know from Day 1. L2
means Euclidean distance instead. Same idea, different ruler.

**IVF (IndexIVFFlat).** IVF stands for inverted file index. Before
searching, it groups all vectors into clusters with k-means, say 100
clusters. At search time it first finds the few clusters closest to the
query, then compares only against the vectors inside those clusters. Most
vectors are never compared, so it is much faster at scale. The price is
that it is approximate: if the true best chunk sits in a cluster you
skipped, you miss it. IVF needs a training step, the clustering, and a
setting called nprobe, which is how many clusters to search. Higher nprobe
means more accurate and slower. Rule of thumb: Flat below 100,000 vectors,
IVF above it.

### Alternatives to FAISS

All of these do the same core job, store vectors and find the nearest
ones. They differ in setup, scale, and extras.

- **Chroma.** A real vector database that runs inside Python, no server.
It stores collections with metadata, so you can filter by source file or
date, and it saves to disk automatically. The easiest next step after
FAISS.
- **Qdrant.** A vector database that can run as a server or in memory. It
is built for production search, with strong filtering and payloads. Good
when your RAG becomes a service.
- **Weaviate.** A vector database server with built-in modules, including
its own vectorizer, and a GraphQL interface. Good when you want the
database to do more of the work.
- **Pinecone.** A managed cloud vector database. No server to run at all,
but it needs an API key and the free tier has limits. Good when you want
zero setup.
- **pgvector.** An extension that turns PostgreSQL into a vector database.
Good when your data already lives in Postgres and you want vectors next to
your normal tables.

For this class, FAISS is the right pick: it installs in one line, needs no
server and no key, and it teaches the core idea, an index plus a search.

In [36]:
# Step 3a: build the FAISS index
import faiss

# normalize: every vector gets length 1, so inner product = cosine similarity
faiss.normalize_L2(chunk_embeddings)
index = faiss.IndexFlatIP(384)
index.add(chunk_embeddings)
print('Index type: IndexFlatIP (Flat, exact search)')
print('Dimension:', index.d)
print('Vectors in the index:', index.ntotal)

Index type: IndexFlatIP (Flat, exact search)
Dimension: 384
Vectors in the index: 1342


In [37]:
# Step 3b: save the index and load it back (this is what makes it a database)
faiss.write_index(index, 'rag.index')
print('Index saved to rag.index')
loaded_index = faiss.read_index('rag.index')
print('Index loaded back. Vectors in the loaded index:', loaded_index.ntotal)

# prove the loaded index gives the same answers
test_q = embed(['What is precision?'])
faiss.normalize_L2(test_q)
scores_a, ids_a = index.search(test_q, 3)
scores_b, ids_b = loaded_index.search(test_q, 3)
print('Same results after save and load:', (ids_a == ids_b).all())

Index saved to rag.index
Index loaded back. Vectors in the loaded index: 1342
Same results after save and load: True


## Retrieval and top-k

Retrieval is one step: embed the question with the same model, then ask
the index for the k nearest chunk vectors. FAISS returns the scores and
the chunk numbers. You look up the chunk text and the file it came from.

k is your choice. Small k means less context for the model, and a higher
risk of missing the chunk that holds the answer. Large k means more
context, but also more noise, and the model has a limited context window.
Day 1 used k equals 3 and k equals 5. You will compare both here, on the
same question, and see the tradeoff with your own eyes.

In [38]:
# Step 4a: the retrieve function, then one question at k=3 and k=5
def retrieve(question, k=5):
    q = embed([question])
    faiss.normalize_L2(q)
    scores, ids = loaded_index.search(q, k)
    results = []
    for i in range(k):
        idx = int(ids[0][i])
        results.append({
            'score': float(scores[0][i]),
            'source': sources[idx],
            'chunk': chunks[idx],
        })
    return results

def show(question, k):
    print(f'Question: {question} (k={k})')
    for rank, hit in enumerate(retrieve(question, k), start=1):
        print(f"\n  Rank {rank} | score {hit['score']:.3f} | from {hit['source']}")
        print('  ' + hit['chunk'][:220].replace('\n', ' '))
    print('\n' + '-' * 70)

show('What is precision?', 3)
show('What is precision?', 5)

Question: What is precision? (k=3)

  Rank 1 | score 0.662 | from 12_evaluation_metrics.txt
  $$\text{Precision} = \frac{TP}{TP + FP}$$  Low precision means lots of false alarms. Use this when a false alarm is expensive: a spam filter that sends real emails to the junk folder annoys people, a security alert that 

  Rank 2 | score 0.593 | from 12_evaluation_metrics.txt
  Mixing these two up is the most common mistake, so say it aloud: precision looks at the model's alerts, recall looks at reality. ### Which One Should You Care About?  

  Rank 3 | score 0.580 | from 12_evaluation_metrics.txt
  1. Reporting only accuracy on lopsided data. 2. Mixing up precision and recall (remember: precision looks at the model's alerts, recall looks at reality). 3. Judging a model on its training data. 4. Declaring a winner fr

----------------------------------------------------------------------
Question: What is precision? (k=5)

  Rank 1 | score 0.662 | from 12_evaluation_metrics.txt
  $$\text{Prec

In [39]:
# Step 4b: two more questions, same pattern
show('What does top-k mean in retrieval?', 3)
show('How do I handle missing values in pandas?', 3)

Question: What does top-k mean in retrieval? (k=3)

  Rank 1 | score 0.716 | from 06_rag_concepts.txt
  Top-k retrieval Retrieval returns the k most similar chunks, called the top-k. A common choice is 3 or 5. More chunks give the model more context, but also more noise, and every extra chunk makes the prompt longer. Choos

  Rank 2 | score 0.643 | from 06_rag_concepts.txt
  the question word "return", so the chunk ranks higher and makes the top-k on its own merit.  •  Multi-step retrieval: round one retrieves the S

  Rank 3 | score 0.457 | from 06_rag_concepts.txt
  stays inside a single chunk, and retrieval returns it complete.  •  

----------------------------------------------------------------------
Question: How do I handle missing values in pandas? (k=3)

  Rank 1 | score 0.695 | from 08_numpy_pandas.txt
  Key Insight: Pandas silently skipped the missing values and averaged only the students who had one, no error, no crash. That is convenient, but also easy to miss, always che

## Step 4c: the split answer test

Now the question from the chunking section. It asks for two facts from
two different files: the SMS spam accuracy, from the fine-tuning notes,
and the definition of RAG, from the RAG guide. Run it at k equals 5 and
check, for each returned chunk, whether it actually contains the fact.

In the run that built this notebook, the combined question returned
neither fact. The focused question about RAG found the definition at
rank 1, so multi-step retrieval, fix number 5 from Day 1, worked for
that half. But the focused accuracy question still did not return the
chunk with 0.991. That is semantic search failing on exact numbers: the
model matches meaning, and a number has almost no meaning to match. The
fixes to try are raising k, or hybrid search, which is Step 8.

In [40]:
# Step 4c: one question asking two things, then two focused questions
def check_facts(question, k=5):
    print(f'Question: {question} (k={k})')
    for rank, hit in enumerate(retrieve(question, k), start=1):
        has_accuracy = '0.991' in hit['chunk']
        has_definition = 'Retrieval-Augmented Generation' in hit['chunk']
        print(f"  Rank {rank} | score {hit['score']:.3f} | from {hit['source']} | "
              f"has accuracy: {has_accuracy} | has RAG definition: {has_definition}")
    print()

check_facts('What accuracy did fine-tuning reach on SMS spam, and what is RAG?', k=5)
check_facts('What accuracy did the fine-tuned model reach on SMS spam?', k=3)
check_facts('What is Retrieval-Augmented Generation?', k=3)

Question: What accuracy did fine-tuning reach on SMS spam, and what is RAG? (k=5)
  Rank 1 | score 0.730 | from 13_sms_spam_finetuning.txt | has accuracy: False | has RAG definition: False
  Rank 2 | score 0.690 | from 13_sms_spam_finetuning.txt | has accuracy: False | has RAG definition: False
  Rank 3 | score 0.687 | from 06_rag_concepts.txt | has accuracy: False | has RAG definition: False
  Rank 4 | score 0.639 | from 13_sms_spam_finetuning.txt | has accuracy: False | has RAG definition: False
  Rank 5 | score 0.627 | from 13_sms_spam_finetuning.txt | has accuracy: False | has RAG definition: False

Question: What accuracy did the fine-tuned model reach on SMS spam? (k=3)
  Rank 1 | score 0.763 | from 13_sms_spam_finetuning.txt | has accuracy: False | has RAG definition: False
  Rank 2 | score 0.737 | from 13_sms_spam_finetuning.txt | has accuracy: False | has RAG definition: False
  Rank 3 | score 0.697 | from 06_rag_concepts.txt | has accuracy: False | has RAG definition: False



## The augmented prompt and grounding

Retrieval alone does not answer anything. The model still has to write the
answer. The augmented prompt is how you hand the retrieved chunks to the
model. It has three parts:

1. **Instruction.** Answer only from the context below. If the answer is
not in the context, say you do not know. Do not guess.
2. **Context.** The retrieved chunks, each labelled with its source file.
3. **Question.** The student's question.

Grounding means the answer is tied to the retrieved text. The instruction
to say "I do not know" is what stops the model from hallucinating when
the documents do not contain the answer. A RAG system that cannot say "I
do not know" is just a search box with extra steps.

Temperature matters here too. You set it low, 0.2, because you want the
model to stick to the context, not get creative.

In [41]:
# Step 5: build the augmented prompt and print it
def build_prompt(question, k=3):
    hits = retrieve(question, k)
    context = ''
    for i, hit in enumerate(hits, start=1):
        context += f"\n[Source {i}: {hit['source']}]\n{hit['chunk']}\n"
    prompt = (
        'Answer the question using ONLY the context below. '
        'If the answer is not in the context, say "I do not know". '
        'Do not guess. Cite the source file for each fact you use.\n'
        f'\nContext:{context}'
        f'\nQuestion: {question}\nAnswer:'
    )
    return prompt

print(build_prompt('What is precision?', k=3))

Answer the question using ONLY the context below. If the answer is not in the context, say "I do not know". Do not guess. Cite the source file for each fact you use.

Context:
[Source 1: 12_evaluation_metrics.txt]
$$\text{Precision} = \frac{TP}{TP + FP}$$

Low precision means lots of false alarms. Use this when a false alarm is expensive: a spam filter that sends real emails to the junk folder annoys people, a security alert that cries wolf gets ignored.

**Recall** (also called sensitivity or true positi

[Source 2: 12_evaluation_metrics.txt]
Mixing these two up is the most common mistake, so say it aloud: precision looks at the model's alerts, recall looks at reality.
### Which One Should You Care About?



[Source 3: 12_evaluation_metrics.txt]
1. Reporting only accuracy on lopsided data.
2. Mixing up precision and recall (remember: precision looks at the model's alerts, recall looks at reality).
3. Judging a model on its training data.
4. Declaring a winner from a tiny difference on

## Generation with DeepSeek

You know DeepSeek from Week 3. The same pattern: an InferenceClient with
your token, a chat call, a system message that sets the behaviour, and the
user message carrying the prompt you just built.

The important comparison in this step is with retrieval against without
retrieval. Same question, twice. Once with the augmented prompt, once with
the bare question and no context. Read both answers and judge them
yourself: which one is grounded, and which one is guessing?

In [43]:
# Step 6: generate locally, with retrieval and without (run on Kaggle)
from transformers import pipeline

pipe = pipeline('text-generation', model='Qwen/Qwen2.5-1.5B-Instruct',
                device_map='auto', torch_dtype='auto')

def ask(prompt):
    out = pipe([{'role': 'user', 'content': prompt}], max_new_tokens=300, do_sample=False)
    return out[0]['generated_text'][-1]['content']

question = 'What is precision?'
prompt_with_rag = build_prompt(question, k=3)
answer_with = ask(prompt_with_rag)
answer_without = ask(question)

print('=== WITH retrieval (RAG) ===')
print(answer_with)
print()
print('=== WITHOUT retrieval (plain prompting) ===')
print(answer_without)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300)

=== WITH retrieval (RAG) ===
Precision is defined as:

\[ \text{Precision} = \frac{TP}{TP + FP} \]

Where:
- \( TP \) represents True Positives,
- \( FP \) represents False Positives.

=== WITHOUT retrieval (plain prompting) ===
Precision refers to the degree of accuracy in measuring or calculating something. It is often used in scientific and technical contexts where it is important to know how close a measurement or calculation is to the true value.

In other words, precision measures how consistently a set of measurements or calculations agree with each other. A high level of precision means that the measurements or calculations are very consistent and reliable, while a low level of precision indicates that they may be less accurate or more variable.

For example, if you measure the length of a table multiple times using a ruler, the precision would refer to how closely those measurements match each other. If your measurements are all within a small range of values, then the precisi

## Test honestly

A RAG system must be tested on questions it should answer AND questions
it should not answer. The second half is the honest half.

Look at the scores first. In the runs done while building this notebook,
questions the documents answer scored around 0.6 to 0.7. The question the
documents do not answer, "What is LangChain?", scored 0.32 at best. That
gap is the signal. A low top score means the documents probably do not
contain the answer, and the model should say "I do not know" instead of
guessing.

Rule of thumb for this corpus: if the top score is below about 0.4, treat
the question as unanswerable from these documents. That number is not
universal, it comes from this corpus and this model, and you should
measure it again on your own corpus. The habit matters more than the
number: check the score before trusting the answer.

In [44]:
# Step 7: the honest test (retrieval part runs here, generation is yours)
print('Question the documents SHOULD answer:')
show('What is the difference between classification and regression?', 3)
print('Question the documents should NOT answer:')
show('What is LangChain?', 3)
print('Now generate the answer for the LangChain question with retrieval,')
print('and check whether it says it does not know. Use Step 6 for that.')

Question the documents SHOULD answer:
Question: What is the difference between classification and regression? (k=3)

  Rank 1 | score 0.729 | from 09_ml_scikit_learn.txt
  ## 3. Classification vs Regression  Supervised learning itself splits into two flavors, based on what kind of answer you are predicting.  | | Classification | Regression | |---|---|---| | 

  Rank 2 | score 0.683 | from 09_ml_scikit_learn.txt
  Notice both are technically predictions, prediction is the general umbrella term for anything supervised learning produces, not a third category alongside these two. Whether it counts as classification or regression 

  Rank 3 | score 0.670 | from 09_ml_scikit_learn.txt
  Whether it counts as classification or regression depends entirely on the shape of the answer, a fixed set of categories, or an open-ended number. Today is a classification problem, Survived is eit

----------------------------------------------------------------------
Question the documents should NOT answer

## Stretch: hybrid search

Semantic search has one weakness: it matches meaning, not exact words. Day
1 Task 2 showed it. The model number BT-440 means nothing to an embedding
model, so semantic search missed the chunk that contained it, even though
the exact characters were right there.

Hybrid search fixes this by combining two retrievers:

- **Semantic search**, FAISS, which you already built. It finds chunks
with similar meaning.
- **Keyword search**, here TF-IDF from scikit-learn, which you know from
Week 2. It finds chunks that contain the exact words.

Each retriever gives every chunk a score. You scale both scores to the
range 0 to 1, add them, and take the top-k of the combined score. A chunk
that both retrievers like wins. A chunk only one of them likes can still
win if its score is high enough.

Try it on the combined question from Step 4c and check the "contains
0.991" line under each result. If the accuracy chunk appears now, hybrid
search did its job. If it still does not appear, the chunk's wording is
too far from the question even for keyword search, and raising k is the
next fix to try. Either way, you have learned where hybrid search helps
and where it does not. That is the whole point of hybrid search: meaning
plus exact words, each covering what the other misses.

In [46]:
# Step 8 (stretch): hybrid search, semantic plus keyword
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer().fit(chunks)
tfidf_matrix = tfidf.transform(chunks)

def hybrid_retrieve(question, k=5, candidates=20):
    # get more candidates from each retriever, then combine and keep the best k
    q_vec = embed([question])
    faiss.normalize_L2(q_vec)
    scores, ids = loaded_index.search(q_vec, candidates)
    sem = {int(ids[0][i]): float(scores[0][i]) for i in range(candidates)}
    key_scores = (tfidf.transform([question]) @ tfidf_matrix.T).toarray()[0]
    key_top = key_scores.argsort()[::-1][:candidates]
    key = {int(i): float(key_scores[i]) for i in key_top}
    combined = {}
    for i, s in sem.items():
        combined[i] = combined.get(i, 0) + s
    max_key = max(key.values()) if key else 1.0
    for i, s in key.items():
        combined[i] = combined.get(i, 0) + (s / max_key)
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]
    return [(score, sources[i], chunks[i]) for i, score in ranked]

print('Hybrid search for the combined question:')
for score, src, ch in hybrid_retrieve(
        'What accuracy did fine-tuning reach on SMS spam, and what is RAG?', k=5):
    print(f"\n  combined {score:.3f} | from {src}")
    print('  ' + ch[:200].replace('\n', ' '))
    print('  contains 0.991:', '0.991' in ch)

Hybrid search for the combined question:

  combined 1.690 | from 13_sms_spam_finetuning.txt
  Fine-Tuning a Model on SMS Spam Week 6, Day 2 Notebook  ---  On Day 1 you learned the theory of fine-tuning. To
  contains 0.991: False

  combined 1.399 | from 06_rag_concepts.txt
  1. Recap: What You Already Know In Week 6 you fine-tuned DistilBERT on SMS spam data. You started from a pre-trained encoder, added a small classification head, and trained with a small learning rate 
  contains 0.991: False

  combined 1.318 | from 13_sms_spam_finetuning.txt
  On Day 1 you learned the theory of fine-tuning. Today you do it for real. You take DistilBERT, a pre-trained model that understands English but has never seen SMS spam, test it first (it guesses like 
  contains 0.991: False

  combined 1.158 | from 13_sms_spam_finetuning.txt
  **Where the information goes.** Right now those numbers encode general English. When we fine-tune on SMS spam, the training nudges them, a little, so they also enco

## Your Turn

Now it is your system. Do these in order, and write down what you observe
after each one.

1. **Add your own document.** Write a short text file about a topic you
know, save it in the `rag_course_docs` folder, rerun Steps 0b to 3, and
ask a question only your file can answer. Does retrieval find it?
2. **Ask your own questions.** Write three questions you know the answers
to from class, and two questions the documents cannot answer. Check the
scores and the generated answers for all five.
3. **Change k.** Run the precision question at k equals 1, 3, 5, and 10.
At what k does the answer stop improving? At what k does noise creep in?
4. **Change the chunk size.** Rebuild with chunk size 150 and again with
600. How do the retrieval scores change? Which size works better for
definition questions?
5. **Stretch.** Your course guides are PDFs. Try adding a PDF to the
corpus with a PDF reader library, and see what the extracted text looks
like. What noise comes with it, and how would you clean it?

## Check Yourself

Answer these without looking back. If you cannot, reread that section.

1. Why can you not embed a whole document at once?
2. In recursive chunking, which separator is tried first, and why?
3. What problem does chunk overlap solve?
4. What does Flat mean in IndexFlatIP, and when is it the right choice?
5. How is IVF different from Flat, and what do you lose in exchange for
speed?
6. Why must the question and the chunks go through the same embedding
model?
7. What is the top score telling you when it is 0.32 instead of 0.66?
8. Name two alternatives to FAISS and say when you would pick each.

## Summary

Today you built a full RAG system. Your own course documents went in as
text files. Recursive chunking cut them into 1,342 pieces. all-MiniLM-L6-v2
turned each piece into a 384 number vector. FAISS stored those vectors in
a real index, an IndexFlatIP, which you saved to disk and loaded back.
Retrieval found the best chunks by cosine similarity. An augmented prompt
handed them to DeepSeek, which wrote a grounded answer with sources. And
you tested it honestly, on questions it should answer and questions it
should not.

The parts stay the same in every RAG system you will meet later. Only the
scale changes: bigger corpora, IVF instead of Flat, Chroma or Qdrant
instead of FAISS, reranking after retrieval. The ideas you used today are
the ideas those systems are built from.

## Key Terms Recap

- **Corpus.** The set of documents the system can read.
- **Chunk.** A small piece of a document, about 300 characters here.
- **Chunk size.** The maximum length of a chunk.
- **Chunk overlap.** The text repeated between neighbouring chunks.
- **Recursive chunking.** Splitting text by trying big separators first,
then smaller ones, only where needed.
- **Split chunk problem.** One fact cut across two chunks, so retrieval
may return only half of it.
- **Embedding.** A vector of numbers that captures the meaning of text.
- **Vector database.** A database that stores vectors and searches them
by similarity.
- **FAISS.** Meta's library for fast vector search, used here as the
vector database.
- **Flat index.** Exact search, compares the query against every vector.
- **IVF index.** Approximate search, compares only the vectors in the
nearest clusters.
- **nprobe.** In IVF, how many clusters to search.
- **Top-k.** The k most similar chunks returned by retrieval.
- **Augmented prompt.** The prompt that combines retrieved context with
the question.
- **Grounding.** Tying the answer to retrieved text, so it can be
checked.
- **Hybrid search.** Combining semantic search with keyword search.